# TP53 and PIK3CA mutation across breast cancer's PAM50 subtypes

**Data kit lab** `tcga-breast-subtype-mutations` · Discovery · about 40 minutes

**The question.** How often are TP53 and PIK3CA (and the other genes named in the same table row) mutated in each PAM50 subtype of breast cancer? Computed from the cBioPortal study `brca_tcga_pub` for the tumours that have mutation, copy-number and expression data, compared with the "DNA Mutations" row of Table 1 of the TCGA Nature 2012 paper.

**What you do.** You read sample lists, a clinical attribute (PAM50 subtype) and five genes' mutation calls through the cBioPortal REST API, and count mutated samples per subtype the way the paper's Table 1 does.

**The data.** cBioPortal study `brca_tcga_pub` (TCGA, Nature 2012). Inside a company, a study like this one is held as sample lists, clinical attributes and molecular profiles in a portal such as cBioPortal, and reached either through its web pages or through the same REST API this notebook calls directly. The data are cBioPortal's and TCGA's, used under cBioPortal's terms (ODbL, with attribution). Nothing is copied into this notebook: the first cell downloads the files from their source (about 0.6 MB, once).

**You need.** Python 3.10 or later with `pandas`.

In [ ]:
# datakit:setup (generated from lab.yaml and expected.json by scripts/datakit/harness.py sync; do not edit by hand)
import hashlib, json, os, pathlib, urllib.request

LAB = 'tcga-breast-subtype-mutations'
FILES = json.loads(r'''{
 "complete": {
  "url": "https://www.cbioportal.org/api/sample-lists/brca_tcga_pub_complete",
  "name": "complete",
  "sha256": "b151e51030d10bb7a0c61b0f3144668f35007048f3667e4ec0586530d074133a",
  "bytes": 8556
 },
 "sequenced": {
  "url": "https://www.cbioportal.org/api/sample-lists/brca_tcga_pub_sequenced",
  "name": "sequenced",
  "sha256": "c97d1cd0c0e162e3dc490e92e082d334c5a457ac08f7ccfff09df416473b12cc",
  "bytes": 9362
 },
 "pam50": {
  "url": "https://www.cbioportal.org/api/studies/brca_tcga_pub/clinical-data?clinicalDataType=SAMPLE&attributeId=PAM50_SUBTYPE&projection=SUMMARY&pageSize=100000",
  "name": "pam50",
  "sha256": "2b2c0652c5c1a18b00af627210ff0bb637d42c622ffc682412f468d248b23824",
  "bytes": 135546
 },
 "tp53": {
  "url": "https://www.cbioportal.org/api/molecular-profiles/brca_tcga_pub_mutations/mutations?sampleListId=brca_tcga_pub_sequenced&entrezGeneId=7157&projection=SUMMARY&pageSize=100000",
  "name": "tp53",
  "sha256": "f4279a88a8ed762c83e6152d1bbea905a0b08c00519e49f9a4f576c9bdec31f6",
  "bytes": 126511
 },
 "pik3ca": {
  "url": "https://www.cbioportal.org/api/molecular-profiles/brca_tcga_pub_mutations/mutations?sampleListId=brca_tcga_pub_sequenced&entrezGeneId=5290&projection=SUMMARY&pageSize=100000",
  "name": "pik3ca",
  "sha256": "4503e5d8dae21bb4e5816d6064493bdb3ce951a52d3233b993338a1e4fe15010",
  "bytes": 127543
 },
 "gata3": {
  "url": "https://www.cbioportal.org/api/molecular-profiles/brca_tcga_pub_mutations/mutations?sampleListId=brca_tcga_pub_sequenced&entrezGeneId=2625&projection=SUMMARY&pageSize=100000",
  "name": "gata3",
  "sha256": "5a3e86a9162c97f0dc280359829db789cb3338b907405248534be9f9d746fa53",
  "bytes": 37161
 },
 "map3k1": {
  "url": "https://www.cbioportal.org/api/molecular-profiles/brca_tcga_pub_mutations/mutations?sampleListId=brca_tcga_pub_sequenced&entrezGeneId=4214&projection=SUMMARY&pageSize=100000",
  "name": "map3k1",
  "sha256": "31e46f26e4933b3365a9809c47aabb96c414634a1fe430f41b3743a92d259934",
  "bytes": 42974
 },
 "pik3r1": {
  "url": "https://www.cbioportal.org/api/molecular-profiles/brca_tcga_pub_mutations/mutations?sampleListId=brca_tcga_pub_sequenced&entrezGeneId=5295&projection=SUMMARY&pageSize=100000",
  "name": "pik3r1",
  "sha256": "31a718595848f8b1e17be6a48f4852428dd3bb3fcf130e1a8e319ca48af42c31",
  "bytes": 8199
 }
}''')
PUBLISHED = json.loads(r'''{
 "n_overlap_patients": {
  "label": "Patients on the overlap list",
  "value": 463,
  "tolerance": {
   "exact": true
  }
 },
 "n_basal_like": {
  "label": "Basal-like tumours on the overlap list",
  "value": 81,
  "tolerance": {
   "exact": true
  }
 },
 "pik3ca_luma_pct": {
  "label": "PIK3CA mutated, Luminal A, % of tumours",
  "value": 49.0,
  "tolerance": {
   "abs": 0.5
  }
 },
 "tp53_luma_pct": {
  "label": "TP53 mutated, Luminal A, % of tumours",
  "value": 12.0,
  "tolerance": {
   "abs": 0.5
  }
 },
 "gata3_luma_pct": {
  "label": "GATA3 mutated, Luminal A, % of tumours",
  "value": 14.0,
  "tolerance": {
   "abs": 0.5
  }
 },
 "map3k1_luma_pct": {
  "label": "MAP3K1 mutated, Luminal A, % of tumours",
  "value": 14.0,
  "tolerance": {
   "abs": 0.5
  }
 },
 "tp53_lumb_pct": {
  "label": "TP53 mutated, Luminal B, % of tumours",
  "value": 32.0,
  "tolerance": {
   "abs": 0.5
  }
 },
 "pik3ca_lumb_pct": {
  "label": "PIK3CA mutated, Luminal B, % of tumours",
  "value": 32.0,
  "tolerance": {
   "abs": 0.5
  }
 },
 "map3k1_lumb_pct": {
  "label": "MAP3K1 mutated, Luminal B, % of tumours",
  "value": 5.0,
  "tolerance": {
   "abs": 0.5
  }
 },
 "tp53_basal_pct": {
  "label": "TP53 mutated, Basal-like, % of tumours",
  "value": 84.0,
  "tolerance": {
   "abs": 0.5
  }
 },
 "pik3ca_basal_pct": {
  "label": "PIK3CA mutated, Basal-like, % of tumours",
  "value": 7.0,
  "tolerance": {
   "abs": 0.5
  }
 },
 "tp53_her2e_pct": {
  "label": "TP53 mutated, HER2E, % of tumours",
  "value": 75.0,
  "tolerance": {
   "abs": 0.5
  }
 },
 "pik3ca_her2e_pct": {
  "label": "PIK3CA mutated, HER2E, % of tumours",
  "value": 42.0,
  "tolerance": {
   "abs": 0.5
  }
 },
 "pik3r1_her2e_pct": {
  "label": "PIK3R1 mutated, HER2E, % of tumours",
  "value": 8.0,
  "tolerance": {
   "abs": 0.5
  }
 }
}''')
PUBLISHED_FROM = 'PMC3465532 full-text XML (no page numbers; the article is pages 61 to 70): Table 1, row "DNA Mutations"; rows "p53 Pathway" and "PIK3CA/PTEN Pathway"; body text, patient and subtype population counts | https://www.ebi.ac.uk/europepmc/webservices/rest/PMC3465532/fullTextXML'
DATA_DIR = pathlib.Path(os.environ.get('DATAKIT_DATA', 'datakit-data/' + LAB))


def data_file(file_id):
    """Path of one dataset file. Downloads it from its source on first use and checks the checksum."""
    f = FILES[file_id]
    path = DATA_DIR / f['name']
    if not path.exists():
        DATA_DIR.mkdir(parents=True, exist_ok=True)
        print('downloading', f['url'])
        urllib.request.urlretrieve(f['url'], path)
    if f['sha256']:
        got = hashlib.sha256(path.read_bytes()).hexdigest()
        if got != f['sha256']:
            print('WARNING:', path.name, 'differs from the copy this lab was checked against (the source file changed)')
    return path


def within(computed, check):
    tol, pub = check['tolerance'], check['value']
    if 'exact' in tol:
        return computed == pub
    limit = tol['abs'] if 'abs' in tol else tol['rel'] * abs(pub)
    return abs(computed - pub) <= limit + 1e-12


def report(results):
    """Print computed against published values and save the computed values for the harness."""
    print('Published values:', PUBLISHED_FROM)
    print(f"{'check':<44}{'published':>12}{'computed':>14}  result")
    for key, check in PUBLISHED.items():
        got = results.get(key)
        shown = '(missing)' if got is None else (f'{got:.4f}' if isinstance(got, float) else str(got))
        verdict = 'missing' if got is None else ('match' if within(got, check) else 'DIFFERENT')
        print(f"{check['label'][:43]:<44}{str(check['value']):>12}{shown:>14}  {verdict}")
    out = pathlib.Path(os.environ.get('DATAKIT_RESULTS', 'results.json'))
    out.write_text(json.dumps(results, indent=1))
    print('saved', out)


## 1. PAM50 subtypes, in one sentence each

- **Luminal A** and **Luminal B**: hormone-receptor-driven subtypes, Luminal A generally slower-growing than Luminal B.
- **HER2-enriched (HER2E)**: driven by amplification of the HER2 gene.
- **Basal-like**: the subtype that overlaps most with triple-negative breast cancer.
- **Normal-like**: a smaller, less well defined group; Table 1 of the paper has no column for it, so this lab leaves it out.

## 2. The population: the "Complete samples" list

cBioPortal holds a study as sample lists (named subsets of samples), clinical attributes (one value per sample or patient) and molecular profiles (mutation, copy-number, expression calls). The API returns each as JSON. This lab's population is the sample list "Complete samples": the samples with mutation, copy-number **and** expression data, which is what Table 1's footnote means by its "466 tumor overlap list" (see section 5).

In [ ]:
import json

import pandas as pd

complete = json.loads(data_file("complete").read_text())["sampleIds"]
n_overlap_patients = len(complete)
print("Complete samples:", n_overlap_patients)


## 3. PAM50 subtype of each sample

A clinical-data query returns one record per sample that has the attribute. Not every sample in the study has a PAM50 call.

In [ ]:
pam50_records = json.loads(data_file("pam50").read_text())
print("records:", len(pam50_records))
pam50 = {r["sampleId"]: r["value"] for r in pam50_records}

from collections import Counter
print(Counter(pam50.values()))


## 4. Subtype sizes on the complete list, and a first check

Table 1 has no Normal-like column, so it is left out below. These four sizes are asserted against the counts the specification derived at the portal's last check, so that a later change in the study would stop this notebook rather than silently change the counts that section 10 compares.

In [ ]:
PORTAL_COMPLETE = {"Luminal A": 209, "Luminal B": 112, "HER2-enriched": 53, "Basal-like": 81}

complete_set = set(complete)
sizes_complete = Counter(pam50.get(s) for s in complete_set)
for subtype, n in PORTAL_COMPLETE.items():
    print(f"{subtype:15s} complete-list size: {sizes_complete[subtype]:4d}  (expected {n})")
    assert sizes_complete[subtype] == n, f"{subtype} complete-list size changed"

n_basal_like = sizes_complete["Basal-like"]


## 5. The trap: the 507-sample "sequenced" list is a different population

cBioPortal also has a sample list "Samples with mutation data" (507 samples): everyone sequenced, whether or not they have copy-number and expression data too. Table 1 is **not** built on this list.

In [ ]:
PORTAL_SEQUENCED = {"Luminal A": 224, "Luminal B": 124, "HER2-enriched": 57, "Basal-like": 93}

sequenced = json.loads(data_file("sequenced").read_text())["sampleIds"]
seq_set = set(sequenced)
sizes_sequenced = Counter(pam50.get(s) for s in seq_set)
print("samples in the sequenced list with no PAM50 value:", sum(1 for s in seq_set if s not in pam50))
for subtype, n in PORTAL_SEQUENCED.items():
    print(f"{subtype:15s} sequenced-list size: {sizes_sequenced[subtype]:4d}  (expected {n})")
    assert sizes_sequenced[subtype] == n, f"{subtype} sequenced-list size changed"

print()
print("The complete list has one sample per patient (463). The paper's Table 1 is based on 466 TUMOURS from 463")
print("PATIENTS — three more tumours than the portal can give us. Section 9 shows what using the wrong list does;")
print("section 10 below works out how much those three missing tumours can move a percentage.")


## 6. One gene's mutation file

Each gene file is a list of mutation rows for the 507 sequenced samples, from the `brca_tcga_pub_mutations` molecular profile. TP53 first.

In [ ]:
tp53_rows = json.loads(data_file("tp53").read_text())
print("TP53 rows:", len(tp53_rows), "| distinct samples:", len({r['sampleId'] for r in tp53_rows}))
print(Counter(r["mutationType"] for r in tp53_rows))
print()
for r in tp53_rows[:3]:
    print(r["sampleId"], r["proteinChange"], r["mutationType"])


## 7. The counting rule: drop synonymous changes

A protein change that ends `=` leaves the protein unchanged (a synonymous, or silent, substitution). The paper's Figure 1 legend says it counts non-silent mutations; the portal classes these particular rows `Splice_Region`, not `Silent`, so filtering on `Variant_Classification == "Silent"` alone would miss them. A sample counts as mutated if it has at least one **non-synonymous** row.

In [ ]:
GENES = ["tp53", "pik3ca", "gata3", "map3k1", "pik3r1"]

gene_rows = {g: json.loads(data_file(g).read_text()) for g in GENES}
mutated = {}
for g in GENES:
    rows = gene_rows[g]
    dropped = [r for r in rows if r["proteinChange"].endswith("=")]
    for r in dropped:
        assert r["mutationType"] == "Splice_Region", f"a dropped {g} row is not Splice_Region: {r}"
    kept = [r for r in rows if not r["proteinChange"].endswith("=")]
    mutated[g] = {r["sampleId"] for r in kept}
    print(f"{g.upper():8s} rows={len(rows):4d}  dropped(synonymous)={len(dropped)}  mutated samples={len(mutated[g]):4d}")


## 8. The twelve cells of Table 1's "DNA Mutations" row

Denominator: every sample of the subtype on the complete list, mutated or not. Table 1's "HER2E" is the portal's "HER2-enriched".

In [ ]:
SUBTYPE_GENES = {
    "Luminal A": ["pik3ca", "tp53", "gata3", "map3k1"],
    "Luminal B": ["tp53", "pik3ca", "map3k1"],
    "Basal-like": ["tp53", "pik3ca"],
    "HER2E": ["tp53", "pik3ca", "pik3r1"],
}
PORTAL_LABEL = {"Luminal A": "Luminal A", "Luminal B": "Luminal B", "Basal-like": "Basal-like", "HER2E": "HER2-enriched"}

cells = []
pct_by_cell = {}
for table_col, genes in SUBTYPE_GENES.items():
    subtype_ids = {s for s in complete_set if pam50.get(s) == PORTAL_LABEL[table_col]}
    n = len(subtype_ids)
    for g in genes:
        k = len(subtype_ids & mutated[g])
        pct = 100 * k / n
        pct_by_cell[(table_col, g)] = pct
        cells.append({"subtype": table_col, "gene": g.upper(), "k": k, "n": n, "pct": round(pct, 2)})

print(pd.DataFrame(cells).to_string(index=False))


## 9. The two traps, shown as extra tables (not checks)

(a) The same twelve cells computed on the 507-sample sequenced list instead of the complete list. (b) The same twelve cells without dropping the synonymous rows.

In [ ]:
def within_half(a, b):
    return abs(a - b) <= 0.5 + 1e-9

PUBLISHED_PCT = {
    ("Luminal A", "pik3ca"): 49, ("Luminal A", "tp53"): 12, ("Luminal A", "gata3"): 14, ("Luminal A", "map3k1"): 14,
    ("Luminal B", "tp53"): 32, ("Luminal B", "pik3ca"): 32, ("Luminal B", "map3k1"): 5,
    ("Basal-like", "tp53"): 84, ("Basal-like", "pik3ca"): 7,
    ("HER2E", "tp53"): 75, ("HER2E", "pik3ca"): 42, ("HER2E", "pik3r1"): 8,
}

main_within = sum(within_half(pct_by_cell[k], v) for k, v in PUBLISHED_PCT.items())
print("main analysis (complete list, synonymous rows dropped): cells within 0.5 of the published value:",
      main_within, "of", len(PUBLISHED_PCT))

# (a) the sequenced list instead of the complete list
seq_within = 0
for (table_col, g), published in PUBLISHED_PCT.items():
    subtype_ids = {s for s in seq_set if pam50.get(s) == PORTAL_LABEL[table_col]}
    pct = 100 * len(subtype_ids & mutated[g]) / len(subtype_ids)
    seq_within += within_half(pct, published)
print("trap (a), 507-sample sequenced list: cells within 0.5:", seq_within, "of", len(PUBLISHED_PCT))

# (b) keep the synonymous rows
mutated_with_syn = {g: {r["sampleId"] for r in gene_rows[g]} for g in GENES}
syn_within = 0
for (table_col, g), published in PUBLISHED_PCT.items():
    subtype_ids = {s for s in complete_set if pam50.get(s) == PORTAL_LABEL[table_col]}
    pct = 100 * len(subtype_ids & mutated_with_syn[g]) / len(subtype_ids)
    syn_within += within_half(pct, published)
print("trap (b), synonymous rows kept: cells within 0.5:", syn_within, "of", len(PUBLISHED_PCT))


## 10. Where the three missing tumours might sit

The paper's text says 466 tumours from 463 patients had data on five platforms; the portal's complete list has 463 samples, one per patient, so three tumours behind the published percentages are not on the portal. Every tumour on that list was exome-sequenced, and the paper's text also gives the sequenced population as 510 tumours from 507 patients — three more tumours than patients there too. The Figure 1 legend gives the sequenced tumours by subtype; comparing that with the portal's own 507-sample sizes suggests where the three extra tumours sit (one Luminal A, two Luminal B). This is an inference: the legend's four groups add up to 501, not 510, and the portal's PAM50 labels may not be the paper's. Until 14 Nov 2026 the cell below turned this into wider tolerances for the two Luminal cells, and it still prints them; the check in section 11 no longer uses them.

In [ ]:
import math

# Figure 1's legend gives the sequenced tumours by subtype. This lab's expected.json copies this sentence from the
# paper by code (extract_expected.py); the published-result file itself is not given to this notebook (only the
# input files are), so the four counts are quoted here exactly as extract_expected.py asserted them.
FIG1_SEQUENCED = {"Luminal A": 225, "Luminal B": 126, "HER2-enriched": 57, "Basal-like": 93}
print("Figure 1 legend, sequenced tumours by subtype:", FIG1_SEQUENCED)
print("portal's 507-sample sequenced list, by subtype:", dict(sizes_sequenced))

def tolerance(port_key, n_complete):
    m = FIG1_SEQUENCED[port_key] - PORTAL_SEQUENCED[port_key]
    return round(math.ceil((0.5 + 100 * m / (n_complete + m)) * 100) / 100, 2)

tol_luma = tolerance("Luminal A", PORTAL_COMPLETE["Luminal A"])
tol_lumb = tolerance("Luminal B", PORTAL_COMPLETE["Luminal B"])
print(f"Luminal A tolerance: {tol_luma} points | Luminal B tolerance: {tol_lumb} points")


## 11. Check against the published table

`PUBLISHED` holds the fourteen values copied by code from Table 1, each with the default tolerance of half a point (half a unit of the last printed digit).

In [ ]:
RESULTS = {
    "n_overlap_patients": int(n_overlap_patients),
    "n_basal_like": int(n_basal_like),
}
CHECK_ID = {
    ("Luminal A", "pik3ca"): "pik3ca_luma_pct", ("Luminal A", "tp53"): "tp53_luma_pct",
    ("Luminal A", "gata3"): "gata3_luma_pct", ("Luminal A", "map3k1"): "map3k1_luma_pct",
    ("Luminal B", "tp53"): "tp53_lumb_pct", ("Luminal B", "pik3ca"): "pik3ca_lumb_pct", ("Luminal B", "map3k1"): "map3k1_lumb_pct",
    ("Basal-like", "tp53"): "tp53_basal_pct", ("Basal-like", "pik3ca"): "pik3ca_basal_pct",
    ("HER2E", "tp53"): "tp53_her2e_pct", ("HER2E", "pik3ca"): "pik3ca_her2e_pct", ("HER2E", "pik3r1"): "pik3r1_her2e_pct",
}
for key, cid in CHECK_ID.items():
    RESULTS[cid] = float(pct_by_cell[key])

report(RESULTS)


## What the result says

Twelve of the fourteen checks reproduce: the two counts and ten of the twelve mutation cells are within half a point. Two Luminal cells are not: TP53 in Luminal A (11.48 against 12) and PIK3CA in Luminal B (31.25 against 32). Section 10 shows one possible reason, three sequenced tumours behind the published percentages that are not on the portal's lists, but it is not established, so the lab is marked not reproduced. REVIEW.txt in the lab folder has the detail.

## Try next

1. Recompute the twelve cells using the 825-sample study (every sample, not a named list). How far off does that get?
2. Add the Normal-like subtype as a thirteenth, uncompared column and see which genes are most and least mutated in it.
3. Look up one of the PAM50-unlabelled samples (in the pam50 dict but with no value) on the cBioPortal website. Does the portal show a reason?